In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_ROOT = Path("../data/raw")

PROJECTS = [
    "eclipse",
    "equinox",
    "lucene",
    "mylyn",
    "pde",
]

In [3]:
def load_project(project_name):
    data_dir = DATA_ROOT / project_name

    ck = pd.read_csv(
        data_dir / "single-version-ck-oo.csv",
        sep=";",
    )

    change = pd.read_csv(
        data_dir / "change-metrics.csv",
        sep=";",
    )

    ck = clean_columns(ck)
    change = clean_columns(change)

    # Each class should occur exactly once.
    assert not ck["classname"].duplicated().any()
    assert not change["classname"].duplicated().any()

    # CK and change datasets should describe the same classes.
    assert set(ck["classname"]) == set(change["classname"])

    # Verify that the bug labels agree between the two datasets.
    label_check = ck[["classname", "bugs"]].merge(
        change[["classname", "bugs"]],
        on="classname",
        suffixes=("_ck", "_change"),
        validate="one_to_one",
    )

    assert (
        label_check["bugs_ck"] ==
        label_check["bugs_change"]
    ).all()

    # Keep the bug labels from CK and remove duplicate labels
    # from the change-metrics dataset.
    change_features = change.drop(
        columns=[
            "bugs",
            "nonTrivialBugs",
            "majorBugs",
            "criticalBugs",
            "highPriorityBugs",
        ]
    )

    merged = ck.merge(
        change_features,
        on="classname",
        how="inner",
        validate="one_to_one",
    )

    # Needed for leave-one-project-out evaluation later.
    merged["project"] = project_name

    # Binary prediction target.
    merged["defective"] = (
        merged["bugs"] > 0
    ).astype(int)

    return merged

In [4]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.rstrip(":")
        .str.strip()
    )
    return df

In [5]:
project_dfs = [
    load_project(project)
    for project in PROJECTS
]

all_df = pd.concat(
    project_dfs,
    ignore_index=True,
)

In [6]:
all_df.groupby("project")["defective"].agg(
    observations="size",
    defective="sum",
    defect_rate="mean",
)

,observations,defective,defect_rate
project,,,
eclipse,997,206,0.206620
equinox,324,129,0.398148
lucene,691,64,0.092619
mylyn,1862,245,0.131579
pde,1497,209,0.139613


In [7]:
CK_FEATURES = [
    "wmc",
    "cbo",
    "dit",
    "lcom",
    "rfc",
    "noc",
    "numberOfLinesOfCode",
    "numberOfMethods",
]

CHANGE_FEATURES = [
    "numberOfVersionsUntil",
    "numberOfAuthorsUntil",
    "linesAddedUntil",
    "maxLinesAddedUntil",
    "avgLinesAddedUntil",
    "linesRemovedUntil",
    "maxLinesRemovedUntil",
    "avgLinesRemovedUntil",
    "codeChurnUntil",
    "maxCodeChurnUntil",
    "avgCodeChurnUntil",
    "ageWithRespectTo",
    "weightedAgeWithRespectTo",
]

FEATURES = CK_FEATURES + CHANGE_FEATURES

print("Number of features:", len(FEATURES))

Number of features: 21


In [8]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid

In [9]:
PARAM_GRID = {
    "n_estimators": [200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5],
    "class_weight": ["balanced", "balanced_subsample"],
}

In [10]:
param_combinations = list(ParameterGrid(PARAM_GRID))

print(
    "Parameter combinations:",
    len(param_combinations),
)

Parameter combinations: 36


In [11]:
def make_random_forest(params):
    return RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
    )

In [12]:
def evaluate_inner_lopo(
    training_df,
    features,
    params,
):
    scores = []

    inner_projects = sorted(
        training_df["project"].unique()
    )

    for validation_project in inner_projects:
        inner_train = training_df[
            training_df["project"] != validation_project
        ]

        inner_validation = training_df[
            training_df["project"] == validation_project
        ]

        X_train = inner_train[features]
        y_train = inner_train["defective"]

        X_validation = inner_validation[features]
        y_validation = inner_validation["defective"]

        model = make_random_forest(params)

        model.fit(
            X_train,
            y_train,
        )

        probabilities = model.predict_proba(
            X_validation
        )[:, 1]

        pr_auc = average_precision_score(
            y_validation,
            probabilities,
        )

        scores.append(pr_auc)

    return np.mean(scores)

In [13]:
def select_best_parameters(
    training_df,
    features,
    param_grid,
):
    combinations = list(ParameterGrid(param_grid))

    results = []
    best_params = None
    best_score = -np.inf

    for i, params in enumerate(combinations, start=1):
        print(
            f"    [{i}/{len(combinations)}] {params}",
            flush=True,
        )

        mean_pr_auc = evaluate_inner_lopo(
            training_df,
            features,
            params,
        )

        results.append({
            **params,
            "mean_pr_auc": mean_pr_auc,
        })

        if mean_pr_auc > best_score:
            best_score = mean_pr_auc
            best_params = params.copy()

    results_df = pd.DataFrame(results)

    return best_params, results_df

In [14]:
def nested_lopo(
    df,
    features,
    param_grid,
):
    outer_results = []
    tuning_results = {}

    outer_projects = sorted(
        df["project"].unique()
    )

    for outer_i, test_project in enumerate(
        outer_projects,
        start=1,
    ):
        print()
        print("=" * 60)
        print(
            f"OUTER [{outer_i}/{len(outer_projects)}] "
            f"Held-out project: {test_project}"
        )
        print("=" * 60)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        print("Selecting hyperparameters...")

        best_params, search_results = (
            select_best_parameters(
                outer_train,
                features,
                param_grid,
            )
        )

        tuning_results[test_project] = search_results

        print()
        print("Best parameters:")
        print(best_params)

        # Retrain using ALL four available training projects.
        model = make_random_forest(best_params)

        model.fit(
            outer_train[features],
            outer_train["defective"],
        )

        # Only now touch the outer test project.
        probabilities = model.predict_proba(
            outer_test[features]
        )[:, 1]

        y_test = outer_test["defective"]

        result = {
            "test_project": test_project,
            "n_train": len(outer_train),
            "n_test": len(outer_test),
            "defect_rate": y_test.mean(),

            "roc_auc": roc_auc_score(
                y_test,
                probabilities,
            ),

            "pr_auc": average_precision_score(
                y_test,
                probabilities,
            ),

            "brier": brier_score_loss(
                y_test,
                probabilities,
            ),

            **{
                f"best_{key}": value
                for key, value in best_params.items()
            },
        }

        outer_results.append(result)

        print()
        print(
            f"OUTER RESULT: "
            f"ROC={result['roc_auc']:.4f}, "
            f"PR={result['pr_auc']:.4f}, "
            f"Brier={result['brier']:.4f}"
        )

    return (
        pd.DataFrame(outer_results),
        tuning_results,
    )

In [15]:
nested_results, tuning_results = nested_lopo(
    all_df,
    FEATURES,
    PARAM_GRID,
)


OUTER [1/5] Held-out project: eclipse
Selecting hyperparameters...
    [1/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
    [2/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 200}
    [3/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 200}
    [4/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
    [5/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 200}
    [6/36] {'class_weight': 'balanced', 'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 200}
    [7/36] {'class_weight': 'balanced', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
    [8/36] {'class_weight': 'balanced', 'max_depth': 1

In [16]:
nested_results[
    [
        "test_project",
        "roc_auc",
        "pr_auc",
        "brier",
        "best_max_depth",
        "best_min_samples_leaf",
        "best_max_features",
        "best_class_weight",
    ]
]

,test_project,roc_auc,pr_auc,brier,best_max_depth,best_min_samples_leaf,best_max_features,best_class_weight
0,eclipse,0.745615,0.525041,0.272843,10.0,5,sqrt,balanced
1,equinox,0.748559,0.694341,0.196754,20.0,5,sqrt,balanced
2,lucene,0.729117,0.337315,0.115551,10.0,5,sqrt,balanced_subsample
3,mylyn,0.708398,0.367381,0.132994,10.0,5,sqrt,balanced
4,pde,0.719672,0.302663,0.175824,NaN,5,sqrt,balanced


In [17]:
nested_results[
    ["roc_auc", "pr_auc", "brier"]
].agg(["mean", "std"])

,roc_auc,pr_auc,brier
mean,0.730272,0.445348,0.178793
std,0.017044,0.163139,0.061794


In [18]:
comparison = pd.DataFrame({
    "Untuned RF": {
        "roc_auc": 0.722437,
        "pr_auc": 0.421754,
        "brier": 0.157174,
    },
    "Nested-tuned RF": nested_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

comparison

,Untuned RF,Nested-tuned RF
roc_auc,0.722437,0.730272
pr_auc,0.421754,0.445348
brier,0.157174,0.178793


In [19]:
logistic_nested_results, logistic_tuning_results = (
    nested_logistic_lopo(all_df)
)

NameError: name 'nested_logistic_lopo' is not defined

In [ ]:
logistic_nested_results, logistic_tuning_results = (
    nested_logistic_lopo(all_df)
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid

LOGISTIC_PARAM_GRID = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "class_weight": [None, "balanced"],
}

LOGISTIC_FEATURE_SETS = {
    "Change": CHANGE_FEATURES,
    "CK + Change": CK_FEATURES + CHANGE_FEATURES,
}

In [ ]:
def make_logistic_model(params):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            **params,
            max_iter=2000,
            random_state=42,
        )),
    ])

In [ ]:
def evaluate_logistic_inner_lopo(
    training_df,
    features,
    params,
):
    scores = []

    for validation_project in sorted(
        training_df["project"].unique()
    ):
        inner_train = training_df[
            training_df["project"] != validation_project
        ]

        inner_validation = training_df[
            training_df["project"] == validation_project
        ]

        model = make_logistic_model(params)

        model.fit(
            inner_train[features],
            inner_train["defective"],
        )

        probabilities = model.predict_proba(
            inner_validation[features]
        )[:, 1]

        scores.append(
            average_precision_score(
                inner_validation["defective"],
                probabilities,
            )
        )

    return np.mean(scores)

In [ ]:
def select_best_logistic_configuration(training_df):
    parameter_combinations = list(
        ParameterGrid(LOGISTIC_PARAM_GRID)
    )

    results = []

    best_score = -np.inf
    best_params = None
    best_feature_set_name = None
    best_features = None

    total = (
        len(LOGISTIC_FEATURE_SETS)
        * len(parameter_combinations)
    )

    current = 0

    for feature_set_name, features in LOGISTIC_FEATURE_SETS.items():
        for params in parameter_combinations:
            current += 1

            print(
                f"    [{current}/{total}] "
                f"{feature_set_name} | {params}",
                flush=True,
            )

            score = evaluate_logistic_inner_lopo(
                training_df,
                features,
                params,
            )

            results.append({
                "feature_set": feature_set_name,
                **params,
                "mean_pr_auc": score,
            })

            if score > best_score:
                best_score = score
                best_params = params.copy()
                best_feature_set_name = feature_set_name
                best_features = features.copy()

    return (
        best_feature_set_name,
        best_features,
        best_params,
        pd.DataFrame(results),
    )

In [ ]:
def nested_logistic_lopo(df):
    outer_results = []
    tuning_results = {}

    outer_projects = sorted(
        df["project"].unique()
    )

    for i, test_project in enumerate(
        outer_projects,
        start=1,
    ):
        print()
        print("=" * 60)
        print(
            f"OUTER [{i}/{len(outer_projects)}] "
            f"Held-out project: {test_project}"
        )
        print("=" * 60)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        (
            feature_set_name,
            features,
            best_params,
            search_results,
        ) = select_best_logistic_configuration(
            outer_train
        )

        tuning_results[test_project] = search_results

        print()
        print("Best feature set:", feature_set_name)
        print("Best parameters:", best_params)

        model = make_logistic_model(
            best_params
        )

        model.fit(
            outer_train[features],
            outer_train["defective"],
        )

        probabilities = model.predict_proba(
            outer_test[features]
        )[:, 1]

        y_test = outer_test["defective"]

        result = {
            "test_project": test_project,
            "feature_set": feature_set_name,
            "C": best_params["C"],
            "class_weight": best_params["class_weight"],

            "roc_auc": roc_auc_score(
                y_test,
                probabilities,
            ),

            "pr_auc": average_precision_score(
                y_test,
                probabilities,
            ),

            "brier": brier_score_loss(
                y_test,
                probabilities,
            ),
        }

        outer_results.append(result)

        print(
            f"OUTER RESULT: "
            f"ROC={result['roc_auc']:.4f}, "
            f"PR={result['pr_auc']:.4f}, "
            f"Brier={result['brier']:.4f}"
        )

    return (
        pd.DataFrame(outer_results),
        tuning_results,
    )

In [ ]:
logistic_nested_results, logistic_tuning_results = (
    nested_logistic_lopo(all_df)
)

In [ ]:
logistic_nested_results[
    ["roc_auc", "pr_auc", "brier"]
].agg(["mean", "std"])

In [ ]:
model_comparison = pd.DataFrame({
    "Nested-tuned RF": nested_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "Nested Logistic": logistic_nested_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

model_comparison

In [ ]:
logistic_nested_results[
    [
        "test_project",
        "feature_set",
        "C",
        "class_weight",
        "roc_auc",
        "pr_auc",
        "brier",
    ]
]

In [ ]:
fold_comparison = (
    nested_results[
        ["test_project", "roc_auc", "pr_auc", "brier"]
    ]
    .merge(
        logistic_nested_results[
            ["test_project", "roc_auc", "pr_auc", "brier"]
        ],
        on="test_project",
        suffixes=("_rf", "_logistic"),
    )
)

fold_comparison["delta_roc"] = (
    fold_comparison["roc_auc_logistic"]
    - fold_comparison["roc_auc_rf"]
)

fold_comparison["delta_pr"] = (
    fold_comparison["pr_auc_logistic"]
    - fold_comparison["pr_auc_rf"]
)

fold_comparison["delta_brier"] = (
    fold_comparison["brier_logistic"]
    - fold_comparison["brier_rf"]
)

fold_comparison


In [ ]:
CK_CHANGE_FEATURES = CK_FEATURES + CHANGE_FEATURES

In [ ]:
def nested_logistic_ck_change(df):
    outer_results = []
    tuning_results = {}

    parameter_combinations = list(
        ParameterGrid(LOGISTIC_PARAM_GRID)
    )

    outer_projects = sorted(df["project"].unique())

    for i, test_project in enumerate(outer_projects, start=1):
        print()
        print("=" * 60)
        print(
            f"OUTER [{i}/{len(outer_projects)}] "
            f"Held-out project: {test_project}"
        )
        print("=" * 60)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        best_score = -np.inf
        best_params = None
        search_results = []

        for j, params in enumerate(
            parameter_combinations,
            start=1,
        ):
            print(
                f"    [{j}/{len(parameter_combinations)}] "
                f"{params}",
                flush=True,
            )

            score = evaluate_logistic_inner_lopo(
                outer_train,
                CK_CHANGE_FEATURES,
                params,
            )

            search_results.append({
                **params,
                "mean_pr_auc": score,
            })

            if score > best_score:
                best_score = score
                best_params = params.copy()

        tuning_results[test_project] = pd.DataFrame(
            search_results
        )

        print("Best parameters:", best_params)

        model = make_logistic_model(best_params)

        model.fit(
            outer_train[CK_CHANGE_FEATURES],
            outer_train["defective"],
        )

        probabilities = model.predict_proba(
            outer_test[CK_CHANGE_FEATURES]
        )[:, 1]

        y_test = outer_test["defective"]

        result = {
            "test_project": test_project,
            "C": best_params["C"],
            "class_weight": best_params["class_weight"],
            "roc_auc": roc_auc_score(
                y_test, probabilities
            ),
            "pr_auc": average_precision_score(
                y_test, probabilities
            ),
            "brier": brier_score_loss(
                y_test, probabilities
            ),
        }

        outer_results.append(result)

        print(
            f"OUTER RESULT: "
            f"ROC={result['roc_auc']:.4f}, "
            f"PR={result['pr_auc']:.4f}, "
            f"Brier={result['brier']:.4f}"
        )

    return (
        pd.DataFrame(outer_results),
        tuning_results,
    )

In [ ]:
logistic_ck_change_results, logistic_ck_change_tuning = (
    nested_logistic_ck_change(all_df)
)

In [ ]:
logistic_ck_change_results[
    ["roc_auc", "pr_auc", "brier"]
].agg(["mean", "std"])

In [ ]:
fair_model_comparison = pd.DataFrame({
    "Random Forest": nested_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "Logistic Regression": logistic_ck_change_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

fair_model_comparison

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

In [ ]:
def fit_project_aware_sigmoid_calibrator(
    model,
    training_df,
    features,
):
    calibration_scores = []
    calibration_targets = []

    projects = sorted(
        training_df["project"].unique()
    )

    for calibration_project in projects:
        calibration_train = training_df[
            training_df["project"] != calibration_project
        ]

        calibration_validation = training_df[
            training_df["project"] == calibration_project
        ]

        fold_model = clone(model)

        fold_model.fit(
            calibration_train[features],
            calibration_train["defective"],
        )

        fold_probabilities = fold_model.predict_proba(
            calibration_validation[features]
        )[:, 1]

        calibration_scores.extend(
            fold_probabilities
        )

        calibration_targets.extend(
            calibration_validation["defective"]
        )

    calibration_scores = np.array(
        calibration_scores
    ).reshape(-1, 1)

    calibration_targets = np.array(
        calibration_targets
    )

    calibrator = LogisticRegression(
        C=1e6,
        max_iter=2000,
    )

    calibrator.fit(
        calibration_scores,
        calibration_targets,
    )

    final_model = clone(model)

    final_model.fit(
        training_df[features],
        training_df["defective"],
    )

    return final_model, calibrator

In [ ]:
def predict_calibrated_probability(
    model,
    calibrator,
    X,
):
    raw_probabilities = model.predict_proba(X)[:, 1]

    calibrated_probabilities = calibrator.predict_proba(
        raw_probabilities.reshape(-1, 1)
    )[:, 1]

    return raw_probabilities, calibrated_probabilities

In [ ]:
def evaluate_logistic_calibration(df):
    results = []

    for test_project in sorted(
        df["project"].unique()
    ):
        print(f"Testing {test_project}...", flush=True)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        selected = logistic_ck_change_results[
            logistic_ck_change_results["test_project"]
            == test_project
        ].iloc[0]

        params = {
            "C": float(selected["C"]),
            "class_weight": selected["class_weight"],
        }

        # Pandas may represent None as NaN
        if pd.isna(params["class_weight"]):
            params["class_weight"] = None

        base_model = make_logistic_model(params)

        model, calibrator = (
            fit_project_aware_sigmoid_calibrator(
                base_model,
                outer_train,
                CK_CHANGE_FEATURES,
            )
        )

        raw_prob, calibrated_prob = (
            predict_calibrated_probability(
                model,
                calibrator,
                outer_test[CK_CHANGE_FEATURES],
            )
        )

        y_test = outer_test["defective"]

        results.append({
            "test_project": test_project,

            "raw_roc": roc_auc_score(
                y_test, raw_prob
            ),
            "cal_roc": roc_auc_score(
                y_test, calibrated_prob
            ),

            "raw_pr": average_precision_score(
                y_test, raw_prob
            ),
            "cal_pr": average_precision_score(
                y_test, calibrated_prob
            ),

            "raw_brier": brier_score_loss(
                y_test, raw_prob
            ),
            "cal_brier": brier_score_loss(
                y_test, calibrated_prob
            ),
        })

    return pd.DataFrame(results)

In [ ]:
logistic_calibration_results = (
    evaluate_logistic_calibration(all_df)
)

logistic_calibration_results

In [ ]:
logistic_calibration_results[
    [
        "raw_roc",
        "cal_roc",
        "raw_pr",
        "cal_pr",
        "raw_brier",
        "cal_brier",
    ]
].mean()

In [ ]:
nested_results.columns


In [ ]:
nested_results

In [ ]:
def make_tuned_rf_from_result(row):
    max_depth = row["best_max_depth"]

    if pd.isna(max_depth):
        max_depth = None
    else:
        max_depth = int(max_depth)

    return RandomForestClassifier(
        n_estimators=int(row["best_n_estimators"]),
        max_depth=max_depth,
        min_samples_leaf=int(row["best_min_samples_leaf"]),
        max_features=row["best_max_features"],
        class_weight=row["best_class_weight"],
        random_state=42,
        n_jobs=-1,
    )

In [ ]:
def evaluate_rf_calibration(df):
    results = []

    for test_project in sorted(
        df["project"].unique()
    ):
        print(
            f"Testing {test_project}...",
            flush=True,
        )

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        selected = nested_results[
            nested_results["test_project"]
            == test_project
        ].iloc[0]

        base_model = make_tuned_rf_from_result(
            selected
        )

        model, calibrator = (
            fit_project_aware_sigmoid_calibrator(
                base_model,
                outer_train,
                CK_CHANGE_FEATURES,
            )
        )

        raw_prob, calibrated_prob = (
            predict_calibrated_probability(
                model,
                calibrator,
                outer_test[CK_CHANGE_FEATURES],
            )
        )

        y_test = outer_test["defective"]

        results.append({
            "test_project": test_project,

            "raw_roc": roc_auc_score(
                y_test,
                raw_prob,
            ),

            "cal_roc": roc_auc_score(
                y_test,
                calibrated_prob,
            ),

            "raw_pr": average_precision_score(
                y_test,
                raw_prob,
            ),

            "cal_pr": average_precision_score(
                y_test,
                calibrated_prob,
            ),

            "raw_brier": brier_score_loss(
                y_test,
                raw_prob,
            ),

            "cal_brier": brier_score_loss(
                y_test,
                calibrated_prob,
            ),
        })

    return pd.DataFrame(results)

In [ ]:
rf_calibration_results = (
    evaluate_rf_calibration(all_df)
)

rf_calibration_results

In [ ]:
rf_calibration_results[
    [
        "raw_roc",
        "cal_roc",
        "raw_pr",
        "cal_pr",
        "raw_brier",
        "cal_brier",
    ]
].mean()

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

In [ ]:
def collect_logistic_calibration_predictions(df):
    predictions = []

    for test_project in sorted(df["project"].unique()):
        print(f"Testing {test_project}...", flush=True)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        selected = logistic_ck_change_results[
            logistic_ck_change_results["test_project"]
            == test_project
        ].iloc[0]

        params = {
            "C": float(selected["C"]),
            "class_weight": selected["class_weight"],
        }

        if pd.isna(params["class_weight"]):
            params["class_weight"] = None

        base_model = make_logistic_model(params)

        model, calibrator = (
            fit_project_aware_sigmoid_calibrator(
                base_model,
                outer_train,
                CK_CHANGE_FEATURES,
            )
        )

        raw_prob, calibrated_prob = (
            predict_calibrated_probability(
                model,
                calibrator,
                outer_test[CK_CHANGE_FEATURES],
            )
        )

        fold_predictions = pd.DataFrame({
            "project": test_project,
            "y_true": outer_test["defective"].to_numpy(),
            "raw_probability": raw_prob,
            "calibrated_probability": calibrated_prob,
        })

        predictions.append(fold_predictions)

    return pd.concat(
        predictions,
        ignore_index=True,
    )

In [ ]:
logistic_predictions = (
    collect_logistic_calibration_predictions(all_df)
)

logistic_predictions.head()

In [ ]:
logistic_predictions.shape

In [ ]:
def collect_rf_calibration_predictions(df):
    predictions = []

    for test_project in sorted(df["project"].unique()):
        print(f"Testing {test_project}...", flush=True)

        outer_train = df[
            df["project"] != test_project
        ]

        outer_test = df[
            df["project"] == test_project
        ]

        selected = nested_results[
            nested_results["test_project"]
            == test_project
        ].iloc[0]

        base_model = make_tuned_rf_from_result(
            selected
        )

        model, calibrator = (
            fit_project_aware_sigmoid_calibrator(
                base_model,
                outer_train,
                CK_CHANGE_FEATURES,
            )
        )

        raw_prob, calibrated_prob = (
            predict_calibrated_probability(
                model,
                calibrator,
                outer_test[CK_CHANGE_FEATURES],
            )
        )

        fold_predictions = pd.DataFrame({
            "project": test_project,
            "y_true": outer_test["defective"].to_numpy(),
            "raw_probability": raw_prob,
            "calibrated_probability": calibrated_prob,
        })

        predictions.append(fold_predictions)

    return pd.concat(
        predictions,
        ignore_index=True,
    )

In [ ]:
rf_predictions = (
    collect_rf_calibration_predictions(all_df)
)

rf_predictions.shape

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

In [ ]:
def plot_calibration_comparison(
    rf_predictions,
    logistic_predictions,
    n_bins=10,
):
    plt.figure(figsize=(8, 7))

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration",
    )

    models = [
        (
            "RF raw",
            rf_predictions,
            "raw_probability",
        ),
        (
            "RF calibrated",
            rf_predictions,
            "calibrated_probability",
        ),
        (
            "Logistic raw",
            logistic_predictions,
            "raw_probability",
        ),
        (
            "Logistic calibrated",
            logistic_predictions,
            "calibrated_probability",
        ),
    ]

    for label, predictions, probability_column in models:
        observed, predicted = calibration_curve(
            predictions["y_true"],
            predictions[probability_column],
            n_bins=n_bins,
            strategy="quantile",
        )

        plt.plot(
            predicted,
            observed,
            marker="o",
            label=label,
        )

    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed defect frequency")
    plt.title(
        "LOPO Bug-Proneness Probability Calibration"
    )

    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
plot_calibration_comparison(
    rf_predictions,
    logistic_predictions,
)

In [ ]:
def plot_project_calibration(
    predictions,
    model_name,
    n_bins=6,
):
    projects = sorted(
        predictions["project"].unique()
    )

    for project in projects:
        project_df = predictions[
            predictions["project"] == project
        ]

        plt.figure(figsize=(6, 5))

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Perfect calibration",
        )

        for label, column in [
            ("Raw", "raw_probability"),
            ("Calibrated", "calibrated_probability"),
        ]:
            observed, predicted = calibration_curve(
                project_df["y_true"],
                project_df[column],
                n_bins=n_bins,
                strategy="quantile",
            )

            plt.plot(
                predicted,
                observed,
                marker="o",
                label=label,
            )

        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed defect frequency")

        plt.title(
            f"{model_name} Calibration - {project}"
        )

        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

In [ ]:
plot_project_calibration(
    logistic_predictions,
    "Logistic Regression",
)

In [ ]:
plot_project_calibration(
    rf_predictions,
    "Random Forest",
)

In [ ]:
def per_project_brier(predictions):
    rows = []

    for project, group in predictions.groupby("project"):
        rows.append({
            "project": project,
            "defect_rate": group["y_true"].mean(),

            "raw_brier": brier_score_loss(
                group["y_true"],
                group["raw_probability"],
            ),

            "calibrated_brier": brier_score_loss(
                group["y_true"],
                group["calibrated_probability"],
            ),
        })

    result = pd.DataFrame(rows)

    result["improvement"] = (
        result["raw_brier"]
        - result["calibrated_brier"]
    )

    return result

In [ ]:
logistic_project_brier = per_project_brier(
    logistic_predictions
)

rf_project_brier = per_project_brier(
    rf_predictions
)

logistic_project_brierrf_project_brier

In [ ]:
rf_project_brier